# Test API — Async (202 + polling) — LOCALHOST
POST returns immediately with 202. Poll `/results/<load_id>` until done.

**Before running:** start the Flask server in a separate terminal with `threaded=True`:
```bash
python main.py
```
Make sure `main.py` ends with `app.run(host='0.0.0.0', port=8080, threaded=True)`.


In [ ]:
import math
import json
import time
from pathlib import Path
import requests
import pandas as pd

# ── Config ────────────────────────────────────────────────────────────────────
BASE_URL      = 'http://localhost:8080'
PAYLOAD_PATH  = './payloads/medicare_input_loadid142.json'
POLL_INTERVAL = 15    # seconds between polls
MAX_WAIT      = 600   # stop after 10 minutes
POST_TIMEOUT  = 120   # large payloads take time to upload + blob-write

# ── Helpers ───────────────────────────────────────────────────────────────────
def sanitize(obj):
    if isinstance(obj, dict):  return {k: sanitize(v) for k, v in obj.items()}
    if isinstance(obj, list):  return [sanitize(v) for v in obj]
    if isinstance(obj, float) and (math.isnan(obj) or math.isinf(obj)): return None
    return obj

# ── 1. Health check ───────────────────────────────────────────────────────────
try:
    r = requests.get(f'{BASE_URL}/', timeout=5)
    print(f'Health: {r.status_code} {r.json()}\n')
except requests.exceptions.ConnectionError:
    raise RuntimeError(
        f'Could not connect to {BASE_URL}. '
        f"Is the Flask server running? Try: python main.py"
    )

# ── 2. Load + POST payload ────────────────────────────────────────────────────
raw  = json.loads(Path(PAYLOAD_PATH).read_text(encoding='utf-8'))
rows = raw['pbp'] if isinstance(raw, dict) and 'pbp' in raw else raw
clean = sanitize(rows)

# Show payload size so you know what to expect
payload_bytes = len(json.dumps(clean))
print(f'Sending {len(clean):,} rows ({payload_bytes/1024/1024:.1f} MB)...')

t_post = time.monotonic()
try:
    r = requests.post(
        f'{BASE_URL}/save',
        json=clean,
        headers={'Content-Type': 'application/json'},
        timeout=POST_TIMEOUT,
    )
except requests.exceptions.ReadTimeout:
    raise RuntimeError(
        f'POST timed out after {POST_TIMEOUT}s. The /save endpoint should return 202 '
        f'almost immediately. Check the Flask terminal for errors. '
        f'Also confirm main.py uses threaded=True in app.run().'
    )
post_time = time.monotonic() - t_post

print(f'HTTP {r.status_code} in {post_time:.2f}s')
if r.status_code != 202:
    print(f'Unexpected status. Response body:')
    print(r.text)
    raise RuntimeError(f'Expected 202, got {r.status_code}')

resp = r.json()
print(json.dumps(resp, indent=2))

load_id   = resp['load_id']
blob_name = resp['blob_name']

# ── 3. Poll /results until done ───────────────────────────────────────────────
print(f'\nPolling /results/{load_id} every {POLL_INTERVAL}s...')
df = None
elapsed = 0
t_proc = time.monotonic()
while elapsed < MAX_WAIT:
    try:
        r = requests.get(f'{BASE_URL}/results/{load_id}', timeout=30)
    except requests.exceptions.ReadTimeout:
        print(f'  [{elapsed:>3}s] poll timed out, retrying...')
        time.sleep(POLL_INTERVAL)
        elapsed += POLL_INTERVAL
        continue

    data = r.json()
    status = data.get('status')
    print(f'  [{elapsed:>3}s] status={status}')

    if status == 'success':
        proc_time = time.monotonic() - t_proc
        print(f'\nDone in {proc_time:.1f}s! {data["result_count"]} rows in blob: {data["blob"]}')
        df = pd.DataFrame(data['results'])
        display(df[['benefitid','benefitname','serviceTypeDesc','tinyDescription']])
        break

    if status == 'error':
        print(f'\nERROR: {data.get("error")}')
        break

    time.sleep(POLL_INTERVAL)
    elapsed += POLL_INTERVAL
else:
    print('Timed out waiting for results.')

# ── 4. Save CSV ───────────────────────────────────────────────────────────────
if df is not None:
    CSV_OUT = f'output_benefits_loadid_{load_id}.csv'
    df.to_csv(CSV_OUT, index=False)
    print(f'\nSaved {len(df)} rows to {CSV_OUT}')
